In [ ]:
import torch
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

In [ ]:
import torch
ground_coarse_token = []
ground_fine_token = []
for i in range(1, 6):
    checkpoint_path = f"/data/esmdiff_target/bpti_kinetic_cluster/bpti_ptm/bpti_{i}.ptm"
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))
    ground_coarse_token.append(checkpoint['small_codebook'])
    ground_fine_token.append(checkpoint['large_codebook'])

In [ ]:
ground_coarse_token

In [ ]:
for j in range(ground_coarse_token[0].shape[0]):
    for i in range(5):
        print(f"{ground_coarse_token[i][j]:>4}", end=" ")
    print()

In [ ]:
stacked_tensor = torch.stack(ground_coarse_token)
data_matrix = stacked_tensor.numpy()
df_view = pd.DataFrame(data_matrix.T, columns=[f"Sample_{i}" for i in range(5)])

print("=== 데이터 형태 (상위 5행) ===")
print(df_view.head())
print("-" * 50)

In [ ]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 5].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 5]

print(f"=== 전체 60개 위치 중 다양성이 높은(5종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

In [ ]:
raw_indices = diverse_rows.index
for i in raw_indices: 
    print(i, end="A+")

In [ ]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 3].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 3]

print(f"=== 전체 60개 위치 중 다양성이 높은(3종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

In [ ]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end="A+")

In [ ]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end=",")

In [ ]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 4].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 4]

print(f"=== 전체 60개 위치 중 다양성이 높은(4종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

In [ ]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end="A+")

In [ ]:
raw_indices = diverse_rows.index

# 2. PDB 매핑을 위한 보정 (+1) 및 리스트 변환
# PDB 파일의 residue 번호가 1번부터 시작한다고 가정할 때:
target_indices = (raw_indices + 1).tolist()

print(f"다양성이 높은 위치(0-based): {raw_indices.tolist()}")
print(f"PDB 시각화용 타겟(1-based): {target_indices}")

In [ ]:
for i in raw_indices:
    print(i, end="A+")

In [ ]:
import py3Dmol
import os

def visualize_pretty_cartoon(pdb_file1, pdb_file2, highlight_residues):
    # 인덱스 안전 변환
    if highlight_residues:
        highlight_residues = [int(x) for x in highlight_residues]
    else:
        highlight_residues = []

    # 파일 읽기
    with open(pdb_file1, 'r') as f: data1 = f.read()
    with open(pdb_file2, 'r') as f: data2 = f.read()

    print(f"=== [구조 1] {os.path.basename(pdb_file1)} ===")
    view1 = py3Dmol.view(width=700, height=500)
    view1.addModel(data1, 'pdb')
    
    # -----------------------------------------------------------
    # [스타일 1] 베이스 스타일 (전체 구조)
    # - arrows: true (Beta sheet 화살표)
    # - style: 'oval' (단면을 둥글게 -> Helix가 두툼해 보임)
    # - color: 'spectrum' (N->C 무지개색) 혹은 단색
    # -----------------------------------------------------------
    view1.setStyle({
        'cartoon': {
            'color': 'white',      # 기본 흰색/회색
            'arrows': True,        # 화살표 켜기 (필수)
            'style': 'oval',       # 단면을 타원형으로 (입체감)
            'thickness': 0.5,      # 기본 두께
            'opacity': 0.7
        }
    })
    
    # -----------------------------------------------------------
    # [스타일 2] 강조 스타일 (Target Residues)
    # - thickness: 1.0 (두껍게)
    # - color: 'red' (빨간색)
    # -----------------------------------------------------------
    if highlight_residues:
        view1.addStyle(
            {'resi': highlight_residues}, 
            {'cartoon': {
                'color': 'red', 
                'arrows': True,    # 여기도 화살표 유지
                'style': 'oval',
                'thickness': 1.0,  # 강조 부분은 더 두껍게!
                'opacity': 1.0
            }}
        )
        
    view1.zoomTo()
    view1.show()

    print(f"\n=== [구조 2] {os.path.basename(pdb_file2)} ===")
    view2 = py3Dmol.view(width=700, height=500)
    view2.addModel(data2, 'pdb')
    
    # 구조 2에도 동일한 스타일 적용 (비교를 위해 파란색 계열로)
    view2.setStyle({
        'cartoon': {
            'color': '#6495ED',    # 옥수수 파란색 (CornflowerBlue)
            'arrows': True,
            'style': 'oval',
            'thickness': 0.5,
            'opacity': 0.7
        }
    })
    
    if highlight_residues:
        view2.addStyle(
            {'resi': highlight_residues}, 
            {'cartoon': {
                'color': 'red', 
                'arrows': True,
                'style': 'oval',
                'thickness': 1.0,
                'opacity': 1.0
            }}
        )

    view2.zoomTo()
    view2.show()

# --- 실행 ---
# target_indices가 정의되어 있어야 합니다.
visualize_pretty_cartoon(
    '/data/esmdiff_target/bpti_kinetic_cluster/bpti_5/bpti_1.pdb', 
    '/data/esmdiff_target/bpti_kinetic_cluster/bpti_5/bpti_3.pdb', 
    target_indices
)